# Step 9 — Model Evaluation
### Credit Card Underwriting Pipeline

---

## Full Evaluation Framework for Credit Models

In the credit industry, model performance is measured by a specific set of metrics
beyond simple accuracy. This notebook covers all of them:

| Metric | Range | Good Credit Model |
|--------|-------|------------------|
| **AUC (ROC)** | 0.5 – 1.0 | > 0.75 |
| **Gini** | 0 – 1.0 | > 0.5 |
| **KS Statistic** | 0 – 1.0 | > 0.35 |
| **SHAP** | — | Consistent with domain knowledge |

---

## Understanding the Metrics

### AUC — Area Under the ROC Curve

> **ROC Curve** plots TPR (True Positive Rate) vs FPR (False Positive Rate) at every possible threshold.
> **AUC** is the area under that curve.
>
> - AUC = 0.5 → model is random (diagonal line)
> - AUC = 1.0 → perfect classifier
> - AUC = 0.80 → for any random pair of one approved and one declined applicant,
>   the model ranks the approved one higher 80% of the time

### Gini Coefficient

> **Gini = 2 × AUC − 1**
> Gini directly measures the model's ability to discriminate.
> - Gini = 0 → random
> - Gini = 1 → perfect

### KS Statistic (Kolmogorov-Smirnov)

> KS measures the maximum gap between the cumulative distribution of scores
> for Approved vs Declined applicants.
> - KS = 0 → model cannot separate the classes at any threshold
> - KS = 1 → model perfectly separates classes
> - KS threshold → the decision threshold that maximises separation

### SHAP — SHapley Additive exPlanations

> SHAP answers: *"Why did the model give this specific applicant this specific score?"*
>
> Derived from cooperative game theory (Shapley values).
> Every feature gets a SHAP value that represents its contribution to that prediction.
> - Positive SHAP → pushed the prediction towards Approved
> - Negative SHAP → pushed the prediction towards Declined
>
> SHAP is increasingly required by regulators (Fair Lending laws, ECOA in the US)
> because it allows the bank to explain exactly why an applicant was declined.


In [ ]:
# ── Full pipeline re-run ─────────────────────────────────────────────────────
import warnings, numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (roc_auc_score, roc_curve, classification_report,
    confusion_matrix, ConfusionMatrixDisplay, accuracy_score)
from imblearn.over_sampling import SMOTE
import shap
warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)
sns.set_theme(style='whitegrid')
SEED=42

DATA_PATH='cc_underwriting_5k_stratified11.csv'
IGNORE_COLS=['applicant_id','target_approved','target_credit_limit_assigned']
df_raw=pd.read_csv(DATA_PATH)
numeric_cols=[c for c in df_raw.select_dtypes(include='number').columns if c not in IGNORE_COLS]
cat_cols=[c for c in df_raw.select_dtypes(include=['object','string']).columns if c not in IGNORE_COLS]
def missing_report(df):
    m=df.isnull().sum();p=m/len(df)*100
    r=pd.DataFrame({'missing_count':m,'missing_pct':p.round(2)})
    return r[r['missing_count']>0].sort_values('missing_pct',ascending=False)
miss=missing_report(df_raw)
drop_cols=miss[miss['missing_pct']>50].index.tolist()
df=df_raw.drop(columns=drop_cols)
numeric_cols=[c for c in numeric_cols if c in df.columns]
cat_cols=[c for c in cat_cols if c in df.columns]
for col in numeric_cols:
    if df[col].isnull().sum()>0: df[col].fillna(df[col].median(),inplace=True)
for col in cat_cols:
    if df[col].isnull().sum()>0: df[col].fillna(df[col].mode()[0],inplace=True)
df['target']=(df['target_approved']=='Yes').astype(int)
df_fe=df.copy();eps=1e-6
df_fe['income_to_limit_ratio']=df_fe['annual_income']/(df_fe['requested_credit_limit']+eps)
df_fe['bureau_score_mean']=df_fe[['fico_score','equifax_score','experian_score','transunion_score']].mean(axis=1)
df_fe['bureau_score_std']=df_fe[['fico_score','equifax_score','experian_score','transunion_score']].std(axis=1)
df_fe['derogatory_x_dti']=df_fe['derogatory_marks_count']*df_fe['debt_to_income_ratio']
df_fe['monthly_net_income']=df_fe['monthly_income']-df_fe['total_monthly_expenses']
for col in ['annual_income','net_worth','total_assets','requested_credit_limit']:
    if col in df_fe.columns: df_fe[col+'_log']=np.log1p(np.clip(df_fe[col],0,None))
le=LabelEncoder()
cat_for_model=[c for c in cat_cols if df_fe[c].nunique()<=30]
for col in cat_for_model: df_fe[col+'_enc']=le.fit_transform(df_fe[col].astype(str))
FINAL_FEATURES=[c for c in numeric_cols+['income_to_limit_ratio','bureau_score_mean',
    'bureau_score_std','derogatory_x_dti','monthly_net_income']+
    [c+'_log' for c in ['annual_income','net_worth','total_assets','requested_credit_limit'] if c+'_log' in df_fe.columns]+
    [c+'_enc' for c in cat_for_model] if c in df_fe.columns]
X=df_fe[FINAL_FEATURES].fillna(0);y=df_fe['target']
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=SEED,stratify=y)
X_train_sm,y_train_sm=SMOTE(random_state=SEED,k_neighbors=5).fit_resample(X_train,y_train)
scaler=StandardScaler()
X_train_sc=scaler.fit_transform(X_train_sm);X_test_sc=scaler.transform(X_test)
X_test_sc_df=pd.DataFrame(X_test_sc,columns=FINAL_FEATURES)
best_rf=RandomForestClassifier(n_estimators=200,max_depth=15,min_samples_leaf=2,
    class_weight='balanced',random_state=SEED,n_jobs=-1)
best_rf.fit(X_train_sc,y_train_sm)
y_pred=best_rf.predict(X_test_sc);y_pred_proba=best_rf.predict_proba(X_test_sc)[:,1]
print('Pipeline complete. Ready for evaluation.')

## 9.1 — AUC, Gini, and KS Metrics

In [ ]:
# Compute core credit model metrics
auc   = roc_auc_score(y_test, y_pred_proba)
gini  = 2 * auc - 1

# ROC curve data
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)

# KS statistic: maximum separation between TPR and FPR at any threshold
ks_values = tpr - fpr
ks        = float(np.max(ks_values))
ks_idx    = np.argmax(ks_values)    # index where KS is maximised
ks_thresh = thresholds[ks_idx]      # the threshold that gives maximum KS

print('=' * 55)
print('     CREDIT MODEL EVALUATION METRICS')
print('=' * 55)
print()
print(f'  AUC  (Area Under ROC)     : {auc:.4f}')
print(f'  Gini (2 x AUC - 1)        : {gini:.4f}')
print(f'  KS   (Max TPR-FPR gap)    : {ks:.4f}')
print(f'  KS   optimal threshold    : {ks_thresh:.3f}')
print()
print('Credit Model Benchmarks:')
print(f'  AUC  > 0.75  : {"PASS" if auc > 0.75 else "FAIL"}  (got {auc:.4f})')
print(f'  Gini > 0.50  : {"PASS" if gini > 0.50 else "FAIL"}  (got {gini:.4f})')
print(f'  KS   > 0.35  : {"PASS" if ks > 0.35 else "FAIL"}  (got {ks:.4f})')

In [ ]:
# Visualise: ROC Curve + KS Chart + Confusion Matrix
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# ── ROC Curve ──────────────────────────────────────────────────────────────────
axes[0].plot(fpr, tpr, color='#2980b9', lw=2.5, label=f'AUC = {auc:.4f}  Gini = {gini:.4f}')
axes[0].plot([0,1],[0,1],'k--', lw=1.2, label='Random classifier (AUC=0.50)')
axes[0].fill_between(fpr, tpr, alpha=0.1, color='#2980b9')
axes[0].scatter(fpr[ks_idx], tpr[ks_idx], color='#e74c3c', zorder=5, s=100,
                label=f'KS point (threshold={ks_thresh:.2f})')
axes[0].set_xlabel('False Positive Rate (FPR)', fontsize=11)
axes[0].set_ylabel('True Positive Rate (TPR)', fontsize=11)
axes[0].set_title('ROC Curve', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# ── KS Chart ──────────────────────────────────────────────────────────────────
# Plot TPR and FPR against threshold; gap between them is KS
n = min(len(thresholds), len(tpr), len(fpr))
axes[1].plot(thresholds[:n], tpr[:n], label='TPR (Sensitivity)', color='#2ecc71', lw=2)
axes[1].plot(thresholds[:n], fpr[:n], label='FPR (1-Specificity)', color='#e74c3c', lw=2)
axes[1].axvline(x=ks_thresh, color='#8e44ad', linestyle='--', lw=1.5,
                label=f'KS = {ks:.4f} at threshold {ks_thresh:.2f}')
# Shade the KS gap
axes[1].fill_between(thresholds[:n], tpr[:n], fpr[:n], alpha=0.08, color='#8e44ad')
axes[1].set_xlabel('Decision Threshold', fontsize=11)
axes[1].set_ylabel('Rate', fontsize=11)
axes[1].set_title('KS Chart\n(Max TPR-FPR gap = KS statistic)', fontsize=13, fontweight='bold')
axes[1].invert_xaxis()
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

# ── Confusion Matrix ──────────────────────────────────────────────────────────
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Declined','Approved'])
disp.plot(ax=axes[2], colorbar=False, cmap='Blues')
axes[2].set_title('Confusion Matrix\n(threshold = 0.50)', fontsize=13, fontweight='bold')

plt.suptitle(f'Model Performance  |  AUC={auc:.4f}  Gini={gini:.4f}  KS={ks:.4f}',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 9.2 — Feature Importance (MDI)

> **MDI (Mean Decrease in Impurity)** measures how much each feature reduces impurity
> (Gini impurity) across all trees and all splits on that feature.
>
> Higher MDI = the model uses this feature more often and it produces cleaner splits.
>
> **Limitation:** MDI can be biased towards high-cardinality features (features with many unique values).
> SHAP values (next section) are more reliable for individual-level explanations.


In [ ]:
# Feature importance from the Random Forest (MDI — Mean Decrease in Impurity)
feat_imp = pd.Series(
    best_rf.feature_importances_,
    index=FINAL_FEATURES
).sort_values(ascending=False)

# Plot top 25
fig, ax = plt.subplots(figsize=(14, 8))
top25 = feat_imp.head(25)
colors_imp = ['#2980b9' if i < 5 else '#7fb3d3' for i in range(len(top25))]
top25.plot(kind='barh', ax=ax, color=colors_imp[::-1], edgecolor='white')
ax.invert_yaxis()
ax.set_xlabel('Mean Decrease in Impurity (MDI)', fontsize=12)
ax.set_title('Top 25 Feature Importances — Random Forest\n(Darker = Top 5 Features)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('Top 10 features by importance:')
for i, (feat, imp) in enumerate(feat_imp.head(10).items(), 1):
    bar = '#' * int(imp * 500)
    print(f'  {i:>2}. {feat:<40} {imp:.4f} {bar}')

## 9.3 — SHAP: Global Feature Importance

> **SHAP Summary Plot (Bar):** Shows the MEAN absolute SHAP value per feature.
> This is the SHAP equivalent of feature importance — tells you which features
> the model relies on most across ALL predictions.
>
> Unlike MDI, SHAP is model-agnostic and does not favour high-cardinality features.


In [ ]:
# SHAP — use a sample for speed (TreeExplainer is still fast, but limit for notebook)
SHAP_SAMPLE = min(500, len(X_test_sc_df))
X_shap = X_test_sc_df.sample(SHAP_SAMPLE, random_state=SEED)

# TreeExplainer is optimised for tree-based models (Random Forest, XGBoost, etc.)
# It computes exact SHAP values efficiently (polynomial time, not exponential)
print('Computing SHAP values...')
explainer   = shap.TreeExplainer(best_rf)
shap_values = explainer.shap_values(X_shap)

# For binary classification, shap_values is a list [class_0_values, class_1_values]
# We want class 1 (Approved) — positive class
shap_vals_pos = shap_values[1] if isinstance(shap_values, list) else shap_values
print('SHAP values computed.')

# SHAP Summary Bar Plot — global feature importance
plt.figure(figsize=(12, 8))
shap.summary_plot(shap_vals_pos, X_shap, plot_type='bar',
                  max_display=20, show=False)
plt.title('SHAP Feature Importance (Mean |SHAP|)\n'
          'Larger bar = feature has bigger impact on predictions',
          fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 9.4 — SHAP: Direction of Feature Impact (Beeswarm)

> **SHAP Beeswarm Plot:** Shows BOTH the magnitude AND direction of each feature's impact.
>
> Each dot = one applicant.
> - **Position on x-axis** = SHAP value (how much this feature pushed the prediction)
> - **Colour** = feature value (red = high, blue = low)
>
> **How to read it:**
> - Red dots on the RIGHT = high values of this feature → pushes towards Approved (good sign)
> - Blue dots on the LEFT = low values → pushes towards Declined
> - Example: `fico_score` — red dots on right means high FICO → more approved (expected!)


In [ ]:
# SHAP Beeswarm — shows direction and magnitude
plt.figure(figsize=(12, 8))
shap.summary_plot(shap_vals_pos, X_shap, max_display=20, show=False)
plt.title('SHAP Beeswarm — Direction of Feature Impact\n'
          'Red dots RIGHT = high feature value pushes towards Approved\n'
          'Blue dots LEFT = low feature value pushes towards Declined',
          fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 9.5 — SHAP: Single Applicant Waterfall Explanation

> **The Waterfall Plot explains ONE individual prediction step by step.**
>
> This is critical for regulatory compliance:
> When a bank declines a credit application under ECOA (Equal Credit Opportunity Act),
> they must be able to explain the reasons. SHAP waterfall provides exactly that.
>
> **How to read the waterfall:**
> - Start at the base value (average model output across all applicants)
> - Each bar shows how much one feature pushed the score UP (red) or DOWN (blue)
> - End at the final prediction probability for this applicant


In [ ]:
# SHAP Waterfall for a single applicant
idx = 0   # which applicant to explain (0 = first test set applicant)

print(f'Explaining applicant index {idx} from the test set:')
print(f'  Actual label     : {"Approved" if y_test.iloc[idx]==1 else "Declined"}')
print(f'  Predicted prob   : {y_pred_proba[idx]:.4f}')
print(f'  Predicted label  : {"Approved" if y_pred[idx]==1 else "Declined"}')
print()

# Extract SHAP values for this one applicant
sv_arr = np.array(shap_vals_pos)

if sv_arr.ndim == 3:
    # shape (n_samples, n_features, n_classes) - pick class 1
    single_shap = sv_arr[idx, :, 1]
elif sv_arr.ndim == 2 and sv_arr.shape[0] == len(X_shap):
    single_shap = sv_arr[idx]
else:
    single_shap = sv_arr[:, 1]

# Get the base value (average model output)
base_val = explainer.expected_value
if isinstance(base_val, (list, np.ndarray)):
    base_val = base_val[1]

# Create SHAP Explanation object for the waterfall plot
explanation = shap.Explanation(
    values       = single_shap,
    base_values  = float(base_val),
    data         = X_shap.iloc[idx].values,
    feature_names= FINAL_FEATURES
)

shap.waterfall_plot(explanation, max_display=15)
plt.title(f'SHAP Waterfall — Applicant {idx}\n'
          f'Red bars = pushed towards Approved | Blue bars = pushed towards Declined',
          fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 9.6 — Scorecard Risk Banding

> **Credit scorecards translate model probabilities into business-friendly risk tiers.**
>
> Banks do not present raw probabilities to their teams. Instead they create:
> - "Risk Bands" (Very High Risk, High Risk, Medium Risk, Low Risk, Excellent)
> - A numeric score (like a FICO score, scaled to 300–850 range)
>
> **Why?** Branch staff, relationship managers, and regulators understand "Low Risk"
> better than "approval_probability = 0.762".


In [ ]:
def prob_to_score(p, base=600, pdo=50, base_odds=1.0):
    """
    Convert approval probability to a scorecard-style numeric score.

    This is the standard logistic scaling used in credit scorecards:
    - base  = score at base_odds (default: 600)
    - pdo   = points to double the odds (default: 50)
    - higher score = lower risk (like FICO)

    Parameters:
        p         : approval probability (0 to 1)
        base      : score at 1:1 odds
        pdo       : points to double the odds
        base_odds : reference odds at the base score
    """
    eps    = 1e-8
    odds   = (1 - p) / (p + eps)   # odds of non-event (declined)
    factor = pdo / np.log(2)
    offset = base - factor * np.log(base_odds)
    return offset + factor * np.log(odds)


# Build a score DataFrame
score_df = pd.DataFrame({
    'prob'  : y_pred_proba,
    'actual': y_test.values
})
score_df['score'] = prob_to_score(score_df['prob'])

# Define risk bands using the credit score scale
bins   = [0, 500, 560, 620, 680, 740, 850]
labels = ['Very High Risk', 'High Risk', 'Medium Risk', 'Low Risk', 'Very Low Risk', 'Excellent']
score_df['risk_band'] = pd.cut(score_df['score'], bins=bins, labels=labels, right=True)

# Summarise each risk band
band_summary = (score_df
    .groupby('risk_band', observed=True)
    .agg(count=('prob','count'), approved=('actual','sum'), avg_score=('score','mean'))
    .assign(approval_rate=lambda x: x['approved']/x['count'])
)

print('Scorecard Risk Band Summary:')
print()
print(f'{"Band":<20} {"Count":>7} {"Approved":>10} {"Approval Rate":>15} {"Avg Score":>12}')
print('-' * 70)
for band, row in band_summary.iterrows():
    print(f'{str(band):<20} {row["count"]:>7.0f} {row["approved"]:>10.0f} '
          f'{row["approval_rate"]:>15.1%} {row["avg_score"]:>12.1f}')

In [ ]:
# Visualise risk band distribution + approval rate
fig, ax = plt.subplots(figsize=(13, 6))
colors_band = ['#c0392b','#e74c3c','#f39c12','#f1c40f','#2ecc71','#1abc9c']

bars = ax.bar(band_summary.index.astype(str), band_summary['count'],
              color=colors_band[:len(band_summary)], edgecolor='white', alpha=0.85)

# Add count labels on bars
for bar, count in zip(bars, band_summary['count']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            str(int(count)), ha='center', fontweight='bold', fontsize=9)

# Overlay approval rate as a line (secondary axis)
ax2 = ax.twinx()
ax2.plot(band_summary.index.astype(str), band_summary['approval_rate'],
         color='#2980b9', marker='o', lw=2.5, markersize=8, label='Approval Rate')
ax2.set_ylabel('Approval Rate', color='#2980b9', fontsize=11)
ax2.set_ylim(0, 1.15)
ax2.tick_params(axis='y', colors='#2980b9')

ax.set_xlabel('Risk Band', fontsize=11)
ax.set_ylabel('Number of Applicants', fontsize=11)
ax.set_title('Risk Band Distribution & Approval Rate\n'
             '(Bars = count per band, Line = approval rate within each band)',
             fontsize=13, fontweight='bold')
ax2.legend(loc='upper right')
plt.tight_layout()
plt.show()

print('A well-calibrated model shows monotonically increasing approval rates')
print('as risk band improves from Very High Risk to Excellent.')

## 9.7 — Final Model Summary

> This is the complete performance report as it would appear in a model validation document.


In [ ]:
print('=' * 60)
print('     CREDIT CARD UNDERWRITING MODEL — FINAL SUMMARY')
print('=' * 60)
print()
print(f'  Dataset shape       : {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns')
print(f'  Final features used : {len(FINAL_FEATURES)}')
print(f'  Train / Test split  : {X_train_sc.shape[0]:,} / {X_test_sc.shape[0]:,}')
print(f'  SMOTE applied       : Yes (training set only)')
print(f'  Scaling             : StandardScaler (fit on train, transform both)')
print(f'  Model               : Random Forest (200 trees, max_depth=15)')
print()
print('  PERFORMANCE METRICS (Test Set):')
print(f'    AUC               : {auc:.4f}  (benchmark: >0.75)')
print(f'    Gini              : {gini:.4f}  (benchmark: >0.50)')
print(f'    KS Statistic      : {ks:.4f}   (benchmark: >0.35)')
print(f'    Accuracy          : {accuracy_score(y_test, y_pred):.4f}')
print()
print('  AUC  benchmark PASS:', 'YES' if auc > 0.75 else 'NO')
print('  Gini benchmark PASS:', 'YES' if gini > 0.50 else 'NO')
print('  KS   benchmark PASS:', 'YES' if ks > 0.35 else 'NO')
print()
print('  EXPLAINABILITY:')
print('    Feature Importance: MDI (Gini impurity reduction)')
print('    Individual Explanation: SHAP TreeExplainer')
print('    Scorecard Banding: 6 risk tiers (Very High Risk to Excellent)')
print()
print('=' * 60)
print('  Pipeline Complete. Model is ready for deployment.')
print('=' * 60)

## Notebook 9 Complete — Full Pipeline Summary

**Congratulations!** You have completed the full Credit Card Underwriting ML Pipeline.

---

### Complete 9-Notebook Summary

| Notebook | Key Action | Output |
|----------|-----------|--------|
| 01 | Problem Understanding & Setup | Libraries, constants |
| 02 | EDA | Target distribution, distributions, correlations |
| 03 | Missing Values | Audit, drop >50%, median/mode imputation |
| 04 | Data Cleaning | Duplicates, outliers, encoding, type fixes |
| 05 | Feature Engineering | WoE/IV, ratios, log transforms, corr filter |
| 06 | Train/Test Split | Stratified 80/20, SMOTE, StandardScaler |
| 07 | Model Selection | RF vs LR vs DT, hyperparameter tuning |
| 08 | Confusion Matrix | TN/FP/FN/TP, threshold analysis |
| 09 | Model Evaluation | AUC, Gini, KS, SHAP, scorecard banding |

### Final Model Performance

| Metric | Result | Benchmark | Status |
|--------|--------|-----------|--------|
| AUC | (see cell above) | > 0.75 | see above |
| Gini | (see cell above) | > 0.50 | see above |
| KS | (see cell above) | > 0.35 | see above |

### What to Do Next

1. **Productionise:** Wrap the model in a FastAPI service (see the FastAPI notebooks)
2. **Monitor:** Track AUC, KS on monthly batches — alert if they degrade (PSI > 0.25)
3. **Retrain:** When performance degrades, retrain on recent data
4. **Fair Lending Review:** Run SHAP analysis across protected classes (gender, race)
   to check for disparate impact
